# Embedding Model Comparison: qwen3-embed vs BGE-M3

Both models via Ollama, compared on ground-truth same/different-topic pairs
(thesis defenses, COVID, condolences, elections — from subject patterns).

In [74]:
import pandas as pd
import numpy as np
import re
import ollama
from itertools import combinations
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

pd.set_option("display.max_colwidth", 80)

## 1. Pull ground-truth pairs from the real corpus

In [75]:
df = pd.read_csv("tovima_nlp_ready.csv", encoding="utf-32", sep="\t")

CATEGORY_PATTERNS = {
    "thesis_defense": r"Διπλωματικ|Διδακτορικ.*Διατριβ",
    "covid": r"covid|COVID|κορωνοϊ|κορονοϊ",
    "condolences": r"υλλυπητ",
    "elections": r"Εκλογ",
}

N_PER_CATEGORY = 6  # keep this small - the point is a cheap, readable check, not a benchmark

samples = {}
for label, pattern in CATEGORY_PATTERNS.items():
    matches = df[df["Subject"].str.contains(pattern, regex=True, na=False)]
    samples[label] = matches.sample(n=min(N_PER_CATEGORY, len(matches)), random_state=42).reset_index(drop=True)
    print(f"{label}: {len(matches)} total in corpus, sampled {len(samples[label])}")

thesis_defense: 1201 total in corpus, sampled 6
covid: 117 total in corpus, sampled 6
condolences: 113 total in corpus, sampled 6
elections: 129 total in corpus, sampled 6


In [76]:
for label, sample in samples.items():
    print(f"--- {label} ---")
    for s in sample["Subject"]:
        print(" ", s[:90])
    print()

--- thesis_defense ---
  [ANNOUNCEMENTS] Δημόσια παρουσίαση της Μεταπτυχιακής Διπλωματικής Εργασίας της μεταπτ
  [TOVIMA] παρουσίαση Διπλωματικής Εργασίας για απόκτηση ΜΔΕ κας Βασιλικής Βαζούρα
  [TOVIMA] Παρουσίαση Μεταπτυχιακής Διπλωματικής Εργασίας
  [ANNOUNCEMENTS] FW: Δημόσια παρουσίαση της Μεταπτυχιακής Διπλωματικής Εργασίας του με
  [TOVIMA] Τμήμα Χημικών Μηχανικών - PhD Defense (Υποστήριξη Διδακτορικής Διατριβής) -
  [TOVIMA] Παρουσιάσεις Μεταπτυχιακών Διπλωματικών Εργασιών για το ΔΠΜΣ στις Περιβαλλοντ

--- covid ---
  [TOVIMA] Στοιχεία COVID-19
  [ANNOUNCEMENTS] Ανακοίνωση - ΜΕΤΡΑ ΠΡΟΣΤΑΣΙΑΣ ΚΑΙΠΡΟΛΗΨΗΣ ΜΕΤΑΔΟΣΗΣ COVID-19
  [TOVIMA] Υπενθύμιση - Ενημέρωση για την διενέργεια Rapid Test για COVID-19 στις εγκατασ
  [TOVIMA] Re: [TOVIMA] Διευκρίνιση σχετικά με τη διενέργεια διαγνωστικών ελέγχων νόση
  [TOVIMA] ΑΝΑΚΟΙΝΩΣΗ ΠΡΟΣ ΤΟΥΣ ΔΙΔΑΣΚΟΝΤΕΣΤΟΥ ΠΑΝΕΠΙΣΤΗΜΙΟΥ ΠΑΤΡΩΝ ΣΤΙΣ ΣΥΝΘΗΚΕΣ ΤΗΣ ΕΠΙΔΗΜ
  [TOVIMA] Ενημέρωση για την διενέργεια Rapid Te

## 2. Build same-topic and different-topic pairs

In [77]:
MAX_PAIRS_PER_TYPE = 15

same_topic_pairs = []
for label, sample in samples.items():
    texts = sample["embedding_text"].tolist()
    for t1, t2 in combinations(texts, 2):
        same_topic_pairs.append((label, label, t1, t2))

diff_topic_pairs = []
labels = list(samples.keys())
for i, label1 in enumerate(labels):
    for label2 in labels[i + 1:]:
        texts1 = samples[label1]["embedding_text"].tolist()
        texts2 = samples[label2]["embedding_text"].tolist()
        for t1 in texts1:
            for t2 in texts2:
                diff_topic_pairs.append((label1, label2, t1, t2))

rng = np.random.default_rng(42)
if len(same_topic_pairs) > MAX_PAIRS_PER_TYPE:
    idx = rng.choice(len(same_topic_pairs), MAX_PAIRS_PER_TYPE, replace=False)
    same_topic_pairs = [same_topic_pairs[i] for i in idx]
if len(diff_topic_pairs) > MAX_PAIRS_PER_TYPE:
    idx = rng.choice(len(diff_topic_pairs), MAX_PAIRS_PER_TYPE, replace=False)
    diff_topic_pairs = [diff_topic_pairs[i] for i in idx]

print(f"{len(same_topic_pairs)} same-topic pairs, {len(diff_topic_pairs)} different-topic pairs")

15 same-topic pairs, 15 different-topic pairs


## 3. Embed every unique text once with each model

In [78]:
all_texts = list({t for pair in same_topic_pairs + diff_topic_pairs for t in (pair[2], pair[3])})
print(f"{len(all_texts)} unique texts to embed")

23 unique texts to embed


In [79]:
MODELS = {
    "qwen3-embed-dimiper": "qwen3-embed-dimiper",
    "bge-m3-dimiper": "bge-m3-dimiper",
}


def embed_all(texts, model_name):
    vectors = {}
    for t in texts:
        response = ollama.embeddings(model=model_name, prompt=t)
        vectors[t] = np.array(response["embedding"])
    return vectors


embeddings_by_model = {}
for friendly_name, ollama_model in MODELS.items():
    print(f"embedding with {ollama_model}...")
    embeddings_by_model[friendly_name] = embed_all(all_texts, ollama_model)
    print(f"  done, vector dim = {len(next(iter(embeddings_by_model[friendly_name].values())))}")

embedding with qwen3-embed-dimiper...
  done, vector dim = 2560
embedding with bge-m3-dimiper...
  done, vector dim = 1024


## 4. Score same-topic vs. different-topic similarity

In [80]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


results = []
for friendly_name in MODELS:
    vectors = embeddings_by_model[friendly_name]

    same_sims = [cosine_sim(vectors[t1], vectors[t2]) for _, _, t1, t2 in same_topic_pairs]
    diff_sims = [cosine_sim(vectors[t1], vectors[t2]) for _, _, t1, t2 in diff_topic_pairs]

    results.append({
        "model": friendly_name,
        "same_topic_mean": np.mean(same_sims),
        "same_topic_min": np.min(same_sims),
        "diff_topic_mean": np.mean(diff_sims),
        "diff_topic_max": np.max(diff_sims),
        "gap": np.mean(same_sims) - np.mean(diff_sims),
        "overlap": max(0, np.max(diff_sims) - np.min(same_sims)),
    })

results_df = pd.DataFrame(results)
results_df

,model,same_topic_mean,same_topic_min,diff_topic_mean,diff_topic_max,gap,overlap
0,qwen3-embed-dimiper,0.630474,0.471779,0.500161,0.650700,0.130313,0.178921
1,bge-m3-dimiper,0.598056,0.528951,0.480562,0.621927,0.117494,0.092976


## 5. Worst-case pairs

In [81]:
for friendly_name in MODELS:
    vectors = embeddings_by_model[friendly_name]
    print(f"=== {friendly_name} ===")

    same_scored = [(cosine_sim(vectors[t1], vectors[t2]), l1, t1, t2) for l1, l2, t1, t2 in same_topic_pairs]
    diff_scored = [(cosine_sim(vectors[t1], vectors[t2]), (l1, l2), t1, t2) for l1, l2, t1, t2 in diff_topic_pairs]

    worst_same = sorted(same_scored)[:2]
    worst_diff = sorted(diff_scored, reverse=True)[:2]

    print("Lowest-scoring SAME-topic pairs (should be high, ideally):")
    for sim, label, t1, t2 in worst_same:
        print(f"  sim={sim:.3f}  [{label}]")
        print(f"    {t1[:80]!r}")
        print(f"    {t2[:80]!r}")

    print("Highest-scoring DIFFERENT-topic pairs (should be low, ideally):")
    for sim, labels, t1, t2 in worst_diff:
        print(f"  sim={sim:.3f}  {labels}")
        print(f"    {t1[:80]!r}")
        print(f"    {t2[:80]!r}")
    print()

=== qwen3-embed-dimiper ===
Lowest-scoring SAME-topic pairs (should be high, ideally):
  sim=0.472  [thesis_defense]
    '[TOVIMA] Παρουσίαση Μεταπτυχιακής Διπλωματικής Εργασίας. ΑΝΑΚΟΙΝΩΣΗ \n\nΣτα πλ'
    '[TOVIMA] Τμήμα Χημικών Μηχανικών - PhD Defense (Υποστήριξη Διδακτορικής Δια'
  sim=0.524  [covid]
    '[TOVIMA] ΑΝΑΚΟΙΝΩΣΗ ΠΡΟΣ ΤΟΥΣ ΔΙΔΑΣΚΟΝΤΕΣΤΟΥ ΠΑΝΕΠΙΣΤΗΜΙΟΥ ΠΑΤΡΩΝ ΣΤΙΣ ΣΥΝΘΗΚΕΣ '
    '[TOVIMA] Ενημέρωση για την διενέργεια Rapid Test για COVID-19 στις εγκαταστάσ'
Highest-scoring DIFFERENT-topic pairs (should be low, ideally):
  sim=0.651  ('covid', 'elections')
    '[ANNOUNCEMENTS] Ανακοίνωση - ΜΕΤΡΑ ΠΡΟΣΤΑΣΙΑΣ ΚΑΙΠΡΟΛΗΨΗΣ ΜΕΤΑΔΟΣΗΣ COVID-19. Ε'
    '[TOVIMA] Με Όραμα και Ρεαλιστικούς Στόχους για το Πανεπιστήμιο του Μέλλοντο'
  sim=0.573  ('covid', 'elections')
    '[TOVIMA] ΑΝΑΚΟΙΝΩΣΗ ΠΡΟΣ ΤΟΥΣ ΔΙΔΑΣΚΟΝΤΕΣΤΟΥ ΠΑΝΕΠΙΣΤΗΜΙΟΥ ΠΑΤΡΩΝ ΣΤΙΣ ΣΥΝΘΗΚΕΣ '
    '[ANNOUNCEMENTS] Αύριο Τρίτη 21-7-2020 Πρυτανικές Εκλογές 9.00 π.μ. έως 6.00'

=== bge-m3-dimiper ===
Lowe

## 6. Decision